In [6]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

print("API 키 로드 완료:", API_KEY[:10], "...")

API 키 로드 완료: AIzaSyAXqC ...


In [10]:
# 수집할 프로그램 정의 (keyword: 검색어, filename: 저장 경로)
PROGRAMS = {
    "yuquiz":     {"keyword": "유퀴즈온더블럭", "filename": "../data/yuquiz.csv"},
    "runningman": {"keyword": "런닝맨",          "filename": "../data/runningman.csv"},
    "nolmyun":    {"keyword": "놀면뭐하니",       "filename": "../data/nolmyun.csv"},
}
MAX_VIDEOS = 10


def search_videos(keyword, max_results=MAX_VIDEOS):
    """키워드로 YouTube 영상 검색 → (video_id, title) 리스트 반환"""
    request = youtube.search().list(
        q=keyword,
        part="snippet",
        type="video",
        maxResults=max_results
    )
    response = request.execute()
    
    results = []
    for item in response["items"]:
        # video 타입만 필터링
        if item["id"]["kind"] == "youtube#video":
            results.append((item["id"]["videoId"], item["snippet"]["title"]))
    
    return results

# 프로그램별 영상 목록 확인
for name, cfg in PROGRAMS.items():
    videos = search_videos(cfg["keyword"])
    print(f"[{name}] 검색된 영상 수: {len(videos)}")

[yuquiz] 검색된 영상 수: 10
[runningman] 검색된 영상 수: 10
[nolmyun] 검색된 영상 수: 9


In [11]:
def get_comments(video_id, video_title, max_comments=200):
    """영상 ID의 댓글을 수집하여 딕셔너리 리스트로 반환. 오류 발생 시 빈 리스트 반환."""
    comments = []
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText"
        )
        response = request.execute()

        while response:
            for item in response["items"]:
                snippet = item["snippet"]["topLevelComment"]["snippet"]
                comments.append({
                    "video_id":    video_id,
                    "video_title": video_title,
                    "author":      snippet["authorDisplayName"],
                    "comment":     snippet["textDisplay"],
                    "likes":       snippet["likeCount"],
                    "date":        snippet["publishedAt"],
                })

            if "nextPageToken" in response and len(comments) < max_comments:
                request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    pageToken=response["nextPageToken"],
                    maxResults=100,
                    textFormat="plainText"
                )
                response = request.execute()
            else:
                break

    except Exception as e:
        print(f"  [스킵] video_id={video_id} | 오류: {e}")

    return comments

In [12]:
# 프로그램별 댓글 수집
results = {}  # {program_name: [comment_dict, ...]}

for name, cfg in PROGRAMS.items():
    print(f"\n{'='*50}")
    print(f"[{name}] 수집 시작 (키워드: {cfg['keyword']})")
    print(f"{'='*50}")

    videos = search_videos(cfg["keyword"])
    program_comments = []

    for video_id, video_title in videos:
        print(f"  수집 중: {video_title[:45]}...")
        comments = get_comments(video_id, video_title)
        program_comments.extend(comments)
        print(f"         → {len(comments)}개 수집")

    results[name] = program_comments
    print(f"\n[{name}] 소계: {len(program_comments)}개")


[yuquiz] 수집 시작 (키워드: 유퀴즈온더블럭)
  수집 중: [예고] 백상 조연상! 명품 배우 유승목🏆 감격의 수상 순간 비하인드와 아내와의 ...
         → 198개 수집
  수집 중: [#유퀴즈온더블럭] 은퇴 후 월 1000만 원 버는 핵심 전략⁉️ 가장 행복한 은...
         → 200개 수집
  수집 중: [#유퀴즈온더블럭] 핵무기 없이도 군사력 5위ㄷㄷ ‘천궁-Ⅱ’ 제작 연구원들이 말...
         → 200개 수집
  수집 중: [#유퀴즈온더블럭] 7년간의 폐섬유증 투병😱 죽음의 문턱까지 다녀온 유열의 이야기...
         → 200개 수집
  수집 중: [#유퀴즈온더블럭] 잠 잘 자는 법=금주⁉️ 20년 불면증 치료 명의가 알려주는 ...
         → 30개 수집
  수집 중: [#유퀴즈온더블럭] 효연: 태연 반응에 긁혔어요💦 소녀시대 태티서의 대항마 효리수...
         → 200개 수집
  수집 중: [예고] 백상 조연상! 명품 배우 유승목🏆 감격의 수상 순간 비하인드와 아내와의 ...
         → 4개 수집
  수집 중: [#유퀴즈온더블럭] 대체 은퇴 설계를 어떻게 했길래🤔 연금술사 김민식 자기님의 노...
         → 172개 수집
  수집 중: [#유퀴즈온더블럭] 이제.. 나도 스타인가?🤔 갑자기 예능 치트키가 되어버린 양상...
         → 200개 수집
  수집 중: [#유퀴즈온더블럭] 7년 만에 기적처럼 돌아온 유열✨ 유언장 쓰며 마음의 준비까지...
         → 12개 수집

[yuquiz] 소계: 1416개

[runningman] 수집 시작 (키워드: 런닝맨)
  수집 중: 발만 보고 어떻게 알아욧!! #런닝맨...
         → 5개 수집
  수집 중: [🔴LIVE] 🏃‍♂️런닝맨 몰아보기🏃‍♂️ | 📺스브스 런닝맨 실시간 스트리밍📢...
  [스킵] video_id=lDcWeklf6DI | 오류: <HttpError 403 when re

In [13]:
# CSV 저장 및 수집 결과 요약
os.makedirs("../data", exist_ok=True)
COLUMNS = ["video_id", "video_title", "author", "comment", "likes", "date"]

print("=" * 50)
print("  수집 결과 요약")
print("=" * 50)

for name, cfg in PROGRAMS.items():
    df = pd.DataFrame(results[name], columns=COLUMNS)
    df.to_csv(cfg["filename"], index=False, encoding="utf-8-sig")
    print(f"  [{name:>10}]  댓글 수: {len(df):>5}개  →  {cfg['filename']}")

print("=" * 50)
print("저장 완료!")

  수집 결과 요약
  [    yuquiz]  댓글 수:  1416개  →  ../data/yuquiz.csv
  [runningman]  댓글 수:   324개  →  ../data/runningman.csv
  [   nolmyun]  댓글 수:  1763개  →  ../data/nolmyun.csv
저장 완료!
